# Building the Core: A Deep Learning Model for Dimension Estimation

In [1]:
# Here the code will construct, train, and evaluate a machine learning model that predicts a product's dimensions (length, width, height) from its image. 

# We will use a powerful technique called transfer learning, which involves adapting a pre-trained Convolutional Neural Network (CNN) for our specific task.
# Here we will use the MobileNetV2 architecture, known for its efficiency and strong performance, implemented with the TensorFlow and Keras libraries.

# MobileNetV2
# - A lightweight CNN architecture developed by Google specifically designed for mobile and embedded vision applications
# - It uses depthwise separable convolutions to reduce the number of parameters and computations.

# Transfer Learning
# - NLP models (and others) are too big and complex to build from scratch and re-train every time.
# - Thus better is start from pre-trained models and fine-tune these models for your own use cases. This is called as Transfer Learning
# - Approaches
# 	- Continue training a pre-trained model (fine-tuning)
# 	- Add new trainable layers to the top of a frozen model
# 	- Retrain from scratch
# 	- Use it as-is

In [2]:
# Prerequistie Installs
# pip install tensorflow pandas numpy scikit-learn pillow

# You should also have the preprocessed data from Step 1, (01_imagemassgeneration.ipynb) specifically:
    # A CSV file (e.g., input_image2mass_images/image2mass_ground_truth.csv) with image identifiers and their corresponding dimensions.
    # A directory (e.g., preprocessed_image2mass_images) containing the normalized image data saved as .npy files.

In [3]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from PIL import Image

In [4]:
# --- 2.1. Configuration, Data (Images) Loading, Convert to NumPy Arrays, Data Splitting ---

In [5]:
# Configuration
DATA_DIR = 'preprocessed_raw_images'
CSV_FILE = 'input_raw_images/raw_product_dataset.csv' # The CSV from Step 1 (01_imagemassgeneration.ipynb) 
IMAGE_DIMS = (224, 224, 3) # Must match the dimensions from preprocessing
TEST_SPLIT_SIZE = 0.2 # 20% of the data is reserved for validation, ensuring the model generalizes well.
RANDOM_STATE = 42
LEARNING_RATE = 0.001 
EPOCHS = 25  # The model trains 25 times over the entire dataset.
BATCH_SIZE = 32 # The dataset will split into mini-batches of 32 samples for training efficiency.

print("Loading dataset...")

# Load the ground truth data
try:
    df = pd.read_csv(CSV_FILE)
    print(f"Loaded {len(df)} entries from {CSV_FILE}")
    # print(df.head())
except FileNotFoundError:
    print(f"Error: The file '{CSV_FILE}' was not found. Please run Step 1 first.")
    exit()

# Prepare file paths and labels
# NOTE: This part assumes your preprocessed files are named `normalized_product_X.npy`
# and correspond to the rows in your CSV. Adjust if your naming is different.
image_files = []
dimensions = []

for index, row in df.iterrows():
    # Construct the expected numpy filename from the preprocessing step
    # npy_filename = os.path.join(DATA_DIR, f"normalized_product_{index + 1}.npy")

    filename = row['imagepath']
    filename = os.path.splitext(filename)[0]
    npy_filename = os.path.join(DATA_DIR, f"normalized_" + filename + ".npy")
    # splitext(filename)[0]

    if os.path.exists(npy_filename):
        image_files.append(npy_filename)
        # We want to predict length, width, and height
        # dimensions.append(row[['length_cm', 'width_cm', 'height_cm']].values)
        dimensions.append(row[['product_length', 'product_width', 'product_height']].values)
    else:
        print(f"Warning: Could not find {npy_filename}. Skipping this entry.")

# Convert to NumPy arrays. Converting data to NumPy arrays in Python is often essential because 
# NumPy is the foundational library for numerical computing—especially in data science, machine learning, and scientific computing.

# When working with images (like in your Image2Mass workflow), converting to NumPy arrays allows:
    # Pixel-level manipulation
    # Feeding data into ML models
    # Efficient resizing, filtering, and normalization

# Example: img = Image.open("sample.jpg")
# img_array = np.array(img)  # Converts to shape (H, W, C)
# In image processing and computer vision, (H, W, C) refers to the shape of an image array:
    # H = Height (number of pixels vertically)
    # W = Width (number of pixels horizontally)
    # C = Channels (number of color components per pixel)

X = np.array([np.load(file) for file in image_files])
y = np.array(dimensions, dtype=np.float32)

print(f"Dataset loaded successfully. Found {len(X)} matching images and labels.")

# Split the data into training and testing sets
# Splitting data into training and testing sets is a foundational practice in machine learning—and it’s absolutely critical for building models that generalize well to unseen data
# Here’s why:
    # The training set is used to teach the model—adjusting weights, learning patterns, and minimizing error.
    # The testing set is used to evaluate how well the model performs on new, unseen data.
    # This split simulates real-world deployment, where the model will encounter data it hasn’t seen before.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT_SIZE, random_state=RANDOM_STATE
)

print(f"Data split into {len(X_train)} training samples and {len(X_test)} testing samples.")
print(f"Data split into {len(y_train)} training samples and {len(y_test)} testing samples.")


Loading dataset...
Loaded 136 entries from input_raw_images/raw_product_dataset.csv
Dataset loaded successfully. Found 136 matching images and labels.
Data split into 108 training samples and 28 testing samples.
Data split into 108 training samples and 28 testing samples.


In [6]:
# --- 2.2. Building the Model with Transfer Learning ---

In [7]:
print("\nBuilding the model...")

# Load the MobileNetV2 base model, pre-trained on ImageNet
# include_top=False means we don't include the final classification layer
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_tensor=Input(shape=IMAGE_DIMS)
)

# Freeze the base model layers to prevent them from being updated during training
base_model.trainable = False

# Add custom layers on top of the base model
# Create the custom regression "head" to place on top of the base model
# This part of the model will be trained.
x = base_model.output
x = GlobalAveragePooling2D()(x) # Averages the spatial features
x = Dense(128, activation='relu')(x)   # A dense layer for learning complex relationships
x = Dense(64, activation='relu')(x)    # Another dense layer
# The final output layer has 3 neurons (for length, width, height) and a linear activation
# because we are predicting continuous values (regression).
predictions = Dense(3, activation='linear')(x)

# Create the final model (Combine the base model and our custom head into the final model)
product_size_prediction_model = Model(inputs=base_model.input, outputs=predictions)

# Display the model's architecture
product_size_prediction_model.summary()


Building the model...


C:\Users\Arun.Manglick\AppData\Local\Temp\ipykernel_58768\2976691297.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,430,403 (9.27 MB)

 Trainable params: 172,419 (673.51 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
# --- 2.3. Compiling the Model ---

In [9]:
print("\nCompiling the model...")
# For regression, 'mean_squared_error' is a common loss function.
# 'mean_absolute_error' gives us a more interpretable metric of how far off our predictions are on average.
product_size_prediction_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='mean_squared_error',
    metrics=['mean_absolute_error']
)


Compiling the model...


In [10]:
# --- 2.4. Training the Model ---

In [11]:
print("\nTraining the model...")
history = product_size_prediction_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# ---------------------------------------------------------------------------------------
# This statement (model.fit) is used to 'TRAIN A DEEP LEARNING MODEL' in TensorFlow/Keras.

# Breakdown of Parameters:
# model.fit(X, y, epochs=25, batch_size=32, validation_split=0.2)

    # working_features_scaled → The input features (X) after scaling.
    # predict_features → The target labels (y) the model is trying to predict.
    # epochs=25 → The model trains 25 times over the entire dataset.
    # batch_size=32 → The dataset is split into mini-batches of 32 samples for training efficiency.
    # validation_split=0.2 → 20% of the data is reserved for validation, ensuring the model generalizes well.

# What Happens During Execution?
# The model iterates 50 times over the dataset to learn patterns.
# It processes 32 samples at a time, updating weights after each batch.
# 20% of the training data is set aside to evaluate performance after each epoch.
# ---------------------------------------------------------------------------------------


Training the model...
Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - loss: 4690.3330 - mean_absolute_error: 39.4302 - val_loss: 2758.4941 - val_mean_absolute_error: 33.1093
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 677ms/step - loss: 3812.8809 - mean_absolute_error: 35.4044 - val_loss: 2453.9226 - val_mean_absolute_error: 30.5846
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 963ms/step - loss: 3682.6875 - mean_absolute_error: 31.6639 - val_loss: 2118.7024 - val_mean_absolute_error: 28.8387
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 738ms/step - loss: 3829.3989 - mean_absolute_error: 33.2849 - val_loss: 1832.1434 - val_mean_absolute_error: 27.9449
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 731ms/step - loss: 2625.1216 - mean_absolute_error: 29.0345 - val_loss: 1686.9108 - val_mean_absolute_error: 29.5631
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 894ms/step - loss: 2122.1326 - mean_absolute_error: 30.5179 - val_loss: 1681.2687 - val_mean_absolute_error: 32.2098
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/s

In [12]:
# --- 2.5. Evaluating the Model ---

In [13]:
print("\nEvaluating model performance...")
loss, mae = product_size_prediction_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Set Mean Absolute Error: {mae:.2f} cm")
print("This means, on average, the model's dimension predictions are off by about {:.2f} cm.".format(mae))

# ---------------------------------------------------------------------------------------
# This statement (model.evaluate) 'EVALUATES THE TRAINED MODEL'S PERFORMANCE' on a given dataset.
# Breakdown:
    # model.evaluate(X_test, y_test) → Runs the trained model on test data and computes the loss & metrics.
    # X_test → The input testing samples
    # y_test → The actual testing samples

# Returns:
    # loss → The error of the model based on the loss function (e.g., binary cross-entropy).
    # accuracy → The accuracy of the model, as defined in model.compile().

# What Happens?
    # The model processes the given dataset without updating weights.
    # It computes the loss based on the selected loss function.
    # It calculates accuracy as a performance metric.
    # The final loss and accuracy values are stored and can be printed.
# ---------------------------------------------------------------------------------------


Evaluating model performance...
Test Set Mean Absolute Error: 31.50 cm
This means, on average, the model's dimension predictions are off by about 31.50 cm.


In [14]:
# --- 2.6. Saving the Model for Future Use ---
product_size_prediction_model.save("02_image_dimension_estimation_model.h5")
print("\nModel saved to '02_image_dimension_estimation_model.h5'.")


Model saved to '02_image_dimension_estimation_model.h5'.


In [15]:
# --- 7. Function Prediction on a New Image ---

In [16]:
def predict_product_dimensions(image_path, model_to_use):
    """
    Takes an image path, preprocesses it, and predicts its dimensions.
    """
    try:
        # Preprocess the new image in the same way as the training data
        img = Image.open(image_path).convert('RGB')
        resized_img = img.resize((IMAGE_DIMS[0], IMAGE_DIMS[1]))
        normalized_array = np.array(resized_img) / 255.0

        # The model expects a batch of images, so we add an extra dimension
        input_data = np.expand_dims(normalized_array, axis=0)

        # Make the prediction
        predicted_dims = model_to_use.predict(input_data)[0]
        return {
            "length_cm": predicted_dims[0],
            "width_cm": predicted_dims[1],
            "height_cm": predicted_dims[2]
        }
    except FileNotFoundError:
        return f"Error: Image not found at {image_path}"
    except Exception as e:
        return f"An error occurred: {e}"

In [17]:
# 2.8 Example usage

In [18]:
image_path = "image_to_predict/81b93e5d.jpg"
predicted_dimensions = predict_product_dimensions(image_path, product_size_prediction_model)
print(predicted_dimensions) 

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
{'length_cm': np.float32(41.776653), 'width_cm': np.float32(25.67067), 'height_cm': np.float32(27.533268)}


In [19]:

# Create a dummy image for prediction if you don't have one.
# Replace 'path/to/your/random/product_image.jpg' with a real image path.
# For this example, let's just use one of our test images.
if len(X_test) > 0:
    # Get the filename corresponding to the first test image
    first_test_index = np.where((X == X_test[0]).all(axis=(1,2,3)))[0][0]
    # original_image_path = os.path.join(
    #     'input_image2mass_images', # Original image folder
    #     f"product_{df.index[first_test_index] + 1}.jpg"
    # )

    original_image_path = os.path.join(
    'input_raw_images', # Original image folder
    f"81a0b666.jpg")

    print(original_image_path)
    
    print("\n--- Example Prediction ---")
    predicted_dimensions = predict_product_dimensions(original_image_path, product_size_prediction_model)

    if isinstance(predicted_dimensions, dict):
        print(f"Image: {original_image_path}")
        print(f"Predicted Dimensions -> Length: {predicted_dimensions['length_cm']:.2f} cm, "
              f"Width: {predicted_dimensions['width_cm']:.2f} cm, "
              f"Height: {predicted_dimensions['height_cm']:.2f} cm")

        # Compare with actual dimensions (y_test)
        actual_dims = y_test[0]
        print(f"Actual Dimensions    -> Length: {actual_dims[0]:.2f} cm, "
              f"Width: {actual_dims[1]:.2f} cm, "
              f"Height: {actual_dims[2]:.2f} cm")

input_raw_images\81a0b666.jpg

--- Example Prediction ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Image: input_raw_images\81a0b666.jpg
Predicted Dimensions -> Length: 100.56 cm, Width: 95.03 cm, Height: 82.93 cm
Actual Dimensions    -> Length: 91.44 cm, Width: 2.49 cm, Height: 2.21 cm
